# Quant Researcher — Practical Lab 4
### Microstructure, Execution & Position Sizing — Part 7 ka practical version


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
np.random.seed(21)
plt.rcParams['figure.figsize']=(8,4)


## A. Market Microstructure Basics — ek simple limit order book simulate karo

Real tick-by-tick NSE order book data free mein nahi milta, isliye ek simplified LOB simulator banate hain — bid/ask levels, spread, mid-price — taaki concepts (spread, depth) khud dikhein.


In [ ]:
def simulate_order_book(mid=100.0, n_levels=5, tick=0.05, base_qty=100):
    bids = pd.DataFrame({
        'price': [round(mid - tick*(i+1), 2) for i in range(n_levels)],
        'qty':   [int(base_qty*np.random.uniform(0.5,1.5)) for _ in range(n_levels)]
    })
    asks = pd.DataFrame({
        'price': [round(mid + tick*(i+1), 2) for i in range(n_levels)],
        'qty':   [int(base_qty*np.random.uniform(0.5,1.5)) for _ in range(n_levels)]
    })
    return bids, asks

bids, asks = simulate_order_book()
best_bid, best_ask = bids['price'].iloc[0], asks['price'].iloc[0]
spread = best_ask - best_bid
print('Best bid:', best_bid, ' Best ask:', best_ask, ' Spread:', round(spread,2))
print('Bid depth (top 5):', bids['qty'].sum(), ' Ask depth (top 5):', asks['qty'].sum())


In [ ]:
# Roll's implied spread estimator — spread nikaalna SIRF price changes se, bina quote data ke
def roll_spread_estimator(price_series):
    price_changes = price_series.diff().dropna()
    autocov = price_changes.autocorr(lag=1) * price_changes.var()
    if autocov >= 0:
        return np.nan   # Roll's model needs negative autocovariance to be valid
    return 2*np.sqrt(-autocov)

# synthetic tick prices bouncing between bid/ask (classic microstructure noise)
n = 2000
true_mid = 100 + np.cumsum(np.random.randn(n)*0.01)
bounce = np.random.choice([-0.025, 0.025], size=n)   # bid-ask bounce
tick_prices = pd.Series(true_mid + bounce)

implied_spread = roll_spread_estimator(tick_prices)
print(f'Roll-implied effective spread: {implied_spread:.4f}  (true round-trip spread was 0.05)')


**Kya dekhna hai:** Roll's estimator koi quote data use nahi karta, sirf price *changes* ka negative autocorrelation — jo bid-ask bounce se hi aata hai. Ye ek classic "live interview" trick question hai: "spread estimate karo bina order book ke."


## B. Execution Algorithms & Market Impact

VWAP vs TWAP scheduling, aur ek simplified Almgren-Chriss style impact model (temporary + permanent impact).


In [ ]:
def twap_schedule(total_qty, n_slices):
    return np.full(n_slices, total_qty / n_slices)

def vwap_schedule(total_qty, volume_profile):
    weights = volume_profile / volume_profile.sum()
    return total_qty * weights

# typical U-shaped intraday volume profile (NSE jaisa — high at open/close, low midday)
intraday_volume = np.array([22,14,10,8,7,7,8,10,14,20], dtype=float)
total_order = 50000

twap = twap_schedule(total_order, 10)
vwap = vwap_schedule(total_order, intraday_volume)

pd.DataFrame({'TWAP': twap, 'VWAP': vwap}, index=[f'slice_{i}' for i in range(10)]).plot(kind='bar', title='TWAP vs VWAP execution schedule')
plt.show()


In [ ]:
def almgren_chriss_impact(qty, adv, daily_vol, eta=0.1, gamma=0.1):
    """eta: temporary impact coefficient, gamma: permanent impact coefficient.
    qty/adv = participation rate -- bigger order relative to average daily volume = bigger impact."""
    participation = qty / adv
    temporary_impact = eta * daily_vol * participation       # reverts after the trade
    permanent_impact = gamma * daily_vol * participation      # stays in the price
    total_cost_bps = (temporary_impact + permanent_impact) * 10000
    return temporary_impact, permanent_impact, total_cost_bps

adv = 2_000_000   # average daily volume (shares)
daily_vol = 0.018  # 1.8% daily volatility

for qty in [5_000, 50_000, 200_000]:
    temp, perm, cost_bps = almgren_chriss_impact(qty, adv, daily_vol)
    print(f'Order size {qty:>7,} ({qty/adv:6.2%} of ADV)  -> est. cost: {cost_bps:6.1f} bps')


**Interpretation:** cost roughly linearly/super-linearly badhta hai participation rate ke saath — yahi reason hai bade orders ko VWAP/TWAP se slice karke, time ke saath, spread kiya jaata hai, ek hi market order mein nahi daala jaata.


## C. Factor Risk Models — portfolio risk ko decompose karna

PCA se statistical factors nikaal ke, ek portfolio ka risk "systematic" (factors se) aur "idiosyncratic" (stock-specific) mein split karte hain.


In [ ]:
from sklearn.decomposition import PCA

n_assets, n_days = 15, 500
asset_returns = pd.DataFrame(np.random.randn(n_days, n_assets)*0.015, columns=[f'A{i}' for i in range(n_assets)])
# inject a common market factor so PCA has something real to find
market_factor = np.random.randn(n_days)*0.012
for col in asset_returns.columns:
    asset_returns[col] += market_factor * np.random.uniform(0.5, 1.5)

pca = PCA(n_components=3).fit(asset_returns)
explained = pca.explained_variance_ratio_
print('Variance explained by top 3 statistical factors:', np.round(explained,3))
print('Factor 1 (likely "market") explains:', f'{explained[0]:.1%}')

weights = np.ones(n_assets)/n_assets   # equal-weight portfolio
portfolio_var = weights @ asset_returns.cov() @ weights
factor_exposure = weights @ pca.components_[0]
systematic_var = (factor_exposure**2) * pca.explained_variance_[0]
idio_var = portfolio_var - systematic_var
print(f'Total portfolio variance : {portfolio_var:.6f}')
print(f'Systematic (factor 1)    : {systematic_var:.6f}  ({systematic_var/portfolio_var:.1%})')
print(f'Idiosyncratic (residual) : {idio_var:.6f}  ({idio_var/portfolio_var:.1%})')


## D. Position Sizing — Kelly Criterion & Risk Budgeting


In [ ]:
def kelly_fraction(win_prob, win_loss_ratio):
    """Fraction of capital to bet. win_loss_ratio = avg win size / avg loss size."""
    return win_prob - (1 - win_prob) / win_loss_ratio

scenarios = [(0.55, 1.0), (0.55, 1.5), (0.50, 2.0), (0.45, 1.0)]
for p, b in scenarios:
    f = kelly_fraction(p, b)
    half_kelly = f/2
    print(f'win_prob={p}, win/loss ratio={b}  -> full Kelly={f:.1%}, HALF Kelly (practitioners prefer this)={half_kelly:.1%}')


In [ ]:
# risk budgeting: allocate a fixed total risk budget across strategies by inverse volatility
strategy_vols = pd.Series({'momentum': 0.12, 'mean_reversion': 0.08, 'pairs': 0.05, 'ml_regime': 0.15})
inv_vol_weights = (1/strategy_vols) / (1/strategy_vols).sum()
print('Risk-budgeted allocation (inverse-vol weighting):')
print(inv_vol_weights.round(3))


**Kyun half-Kelly:** full Kelly theoretically optimal hai *agar* win_prob aur win/loss ratio exactly sahi maalum hon — real life mein estimates galat hote hain, aur Kelly errors ko bahut punish karta hai (drawdowns badi hoti hain). Half-Kelly growth rate ka 75% deta hai lekin volatility aadhi — yahi tradeoff practitioners actually use karte hain.


## Wrap-up

Roll's spread estimator, VWAP/TWAP scheduling, Almgren-Chriss impact, PCA factor decomposition, Kelly/risk-budgeting — ye sab Part 7 ke concepts ko ab number aur code se justify kar sakte ho, sirf definition se nahi.
